# EDA - Phân tích Wrong Cases
Notebook này hiển thị ngẫu nhiên 5 trường hợp False Positive (FP) và 5 trường hợp False Negative (FN) có độ tự tin cao nhất (dựa vào `wrong_case.json` do `extract_wrong_cases.py` sinh ra).

In [21]:
import json
import random
import pandas as pd

# Đọc dữ liệu từ file JSON
with open('wrong_case.json', 'r', encoding='utf-8') as f:
    wrong_cases = json.load(f)

fp_cases = wrong_cases['FP']
fn_cases = wrong_cases['FN']

print(f"Total False Positives (FP): {len(fp_cases)}")
print(f"Total False Negatives (FN): {len(fn_cases)}")

Total False Positives (FP): 53
Total False Negatives (FN): 24


### False Positives (FP)
Model dự đoán là Clickbait (1), nhưng nhãn thực tế là Non-Clickbait (0).

In [22]:
print("Random 5 False Positives:")
for case in random.sample(fp_cases, min(5, len(fp_cases))):
    print("-" * 80)
    print(f"Index: {case['index']} | Predicted Score (Confidence): {case['score']:.4f}")
    print(f"Title: {case['title']}")
    print(f"Lead : {case['lead']}")

Random 5 False Positives:
--------------------------------------------------------------------------------
Index: 360 | Predicted Score (Confidence): 0.5091
Title: Đĩa dồi 6 miếng được bán với giá 500.000 đồng tại lễ hội hoa anh đào Hàn Quốc
Lead : Nhiều người cho rằng, đây là mức giá quá cao cho một món ăn đường phố vốn được coi là rẻ tiền ở Hàn Quốc.
--------------------------------------------------------------------------------
Index: 107 | Predicted Score (Confidence): 0.6849
Title: Ngày mới với tin tức sức khỏe: Cách ngủ máy lạnh không bị khô họng
Lead : 'Việc sử dụng máy lạnh sai cách có thể khiến cơ thể bị tổn thương, đặc biệt là vùng cổ họng và đường hô hấp trên'. Hãy bắt đầu ngày mới với tin tức sức khỏe để xem thêm nội dung bài viết này!
--------------------------------------------------------------------------------
Index: 146 | Predicted Score (Confidence): 0.6648
Title: Cung đường đẹp như tranh cách Hà Nội 3 giờ lái xe, du khách 'đội nắng' săn ảnh
Lead : Gần trưa, khi nắn

### False Negatives (FN)
Model dự đoán là Non-Clickbait (0), nhưng nhãn thực tế là Clickbait (1).

In [23]:
print("Random 5 False Negatives:")
for case in random.sample(fn_cases, min(5, len(fn_cases))):
    print("-" * 80)
    print(f"Index: {case['index']} | Predicted Score (Confidence): {case['score']:.4f}")
    print(f"Title: {case['title']}")
    print(f"Lead : {case['lead']}")

Random 5 False Negatives:
--------------------------------------------------------------------------------
Index: 152 | Predicted Score (Confidence): 0.1039
Title: Từ EDM Festival đến mega concert Kpop đưa G-Dragon về Việt Nam: Ravolution Asia bùng nổ bước chuyển mình chiến lược
Lead : Từ những mùa lễ hội âm nhạc điện tử tiên phong đến siêu concert Kpop, Ravolution Asia đang viết nên hành trình đưa trải nghiệm âm nhạc Việt tiệm cận chuẩn mực toàn cầu.
--------------------------------------------------------------------------------
Index: 362 | Predicted Score (Confidence): 0.3154
Title: Người tuổi này nên chọn nhà hướng Đông Tứ Trạch giúp thu hút tài lộc, gia đạo thuận hòa và sức khỏe dồi dào - Chuyên trang Gia Đình & Xã Hội - Báo Sức khỏe & Đời sống
Lead : Theo phong thủy truyền thống, Đông Tứ Trạch tượng trưng cho nhóm hướng tốt đối với người thuộc mệnh Đông Tứ Mệnh, giúp thu hút tài lộc, gia đạo thuận hòa và sức khỏe dồi dào.
---------------------------------------------------------

### Kiểm tra Conflict Labels (Cùng Title nhưng khác Label)
Tìm các trường hợp trong dữ liệu gốc có tiêu đề giống hệt nhau nhưng bị gán nhãn khác nhau.

In [24]:
raw_df = pd.read_csv('../raw/clickbait_dataset_vietnamese.csv')

# Tìm các title xuất hiện với nhiều hơn 1 label
conflicts = raw_df.groupby('title')['label'].nunique()
conflict_titles = conflicts[conflicts > 1].index

print(f"Số lượng title có nhãn mâu thuẫn: {len(conflict_titles)}")

if len(conflict_titles) > 0:
    # Hiển thị các case mâu thuẫn
    df_conflicts = raw_df[raw_df['title'].isin(conflict_titles)].sort_values('title')
    display(df_conflicts[['title', 'label', 'source']])

Số lượng title có nhãn mâu thuẫn: 0


In [28]:
df = pd.read_csv('../raw/clickbait_dataset_vietnamese.csv')

In [34]:
pd.set_option('display.max_colwidth', None)

In [35]:
temp = df[df['title'] == 'Microsoft chặn trình duyệt Chrome']
print("-" * 80)
print(f"Title: {temp['title']}")
print(f"Lead : {temp['lead_paragraph']}")
print(f"Label : {temp['label']}")

--------------------------------------------------------------------------------
Title: 1530    Microsoft chặn trình duyệt Chrome
Name: title, dtype: object
Lead : 1530    Tính năng của Microsoft khiến Google Chrome “tê liệt“ suốt nhiều tuần, lỗi vô tình hay “chiêu trò“?
Name: lead_paragraph, dtype: object
Label : 1530    non-clickbait
Name: label, dtype: object


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3414 entries, 0 to 3413
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                3414 non-null   object
 1   url               3414 non-null   object
 2   title             3414 non-null   object
 3   lead_paragraph    3411 non-null   object
 4   category          3414 non-null   object
 5   publish_datetime  3383 non-null   object
 6   source            3414 non-null   object
 7   thumbnail_url     3410 non-null   object
 8   label             3414 non-null   object
dtypes: object(9)
memory usage: 240.2+ KB
